In [1]:
!apt-get update

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 90 not upgraded.


In [3]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
ollama_model_id = "qwen2.5:7b"
ollama_port = 11501

In [5]:
!pkill -f "ollama" || true

^C


In [6]:
!nohup bash -c "OLLAMA_HOST=0.0.0.0:11501 OLLAMA_ORIGINS=* ollama serve" > /content/nohup.out 2>&1 &

In [7]:
!sleep 5 && tail -n 50 /content/nohup.out

time=2026-06-21T22:20:26.830Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11501 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* 

In [8]:
!curl http://127.0.0.1:11501/api/tags

{"models":[{"name":"glm-5.2:cloud","model":"glm-5.2:cloud","remote_model":"glm-5.2","remote_host":"https://ollama.com:443","modified_at":"2026-06-21T22:12:40.297654975Z","size":338,"digest":"ce8fd6f94793a6786f6858717b38067feba631861e3b35e8122cb6635004cd4a","details":{"parent_model":"","format":"","family":"glm","families":["glm"],"parameter_size":"756b","quantization_level":"","context_length":1000000},"capabilities":["completion","tools","thinking"]}]}

In [9]:
!OLLAMA_HOST=http://127.0.0.1:11501 ollama pull qwen2.5:7b

In [10]:
!curl http://127.0.0.1:11501/api/tags

{"models":[{"name":"qwen2.5:7b","model":"qwen2.5:7b","modified_at":"2026-06-21T22:23:15.026193444Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584},"capabilities":["completion","tools"]},{"name":"glm-5.2:cloud","model":"glm-5.2:cloud","remote_model":"glm-5.2","remote_host":"https://ollama.com:443","modified_at":"2026-06-21T22:12:40.297654975Z","size":338,"digest":"ce8fd6f94793a6786f6858717b38067feba631861e3b35e8122cb6635004cd4a","details":{"parent_model":"","format":"","family":"glm","families":["glm"],"parameter_size":"756b","quantization_level":"","context_length":1000000},"capabilities":["completion","tools","thinking"]}]}

In [11]:
%%bash
curl -s http://127.0.0.1:11501/api/chat \
  -H "Content-Type: application/json" \
  -d '{
    "model": "qwen2.5:7b",
    "stream": false,
    "messages": [
      {
        "role": "user",
        "content": "What is RAG? Explain briefly."
      }
    ]
  }' | jq -r '.message.content'

RAG stands for Retrieval-Augmented Generation. It's an approach used in natural language processing (NLP) and artificial intelligence to improve the ability of models to generate text or respond to queries more accurately and contextually.

In a RAG system, the process involves two main steps:

1. **Retrieval**: The model retrieves relevant documents or passages from a database (referred to as a knowledge base). This step helps to gather contextual information that can be used to inform the generation of responses or text.

2. **Generation**: Using the retrieved content as input, the model generates a response or continues the text. By incorporating the retrieved context, this approach often results in more accurate and relevant outputs compared to models that generate text based on just the query or prompt without additional context.

RAG systems can be used for various applications such as question answering, document summarization, conversational agents, and more. The key advantage 

In [12]:
!pip install pyngrok

In [13]:
from google.colab import userdata
from pyngrok import ngrok, conf

ngrok_auth = userdata.get('ngrok')

conf.get_default().auth_token = ngrok_auth

port = "11501"

public_url = ngrok.connect(port).public_url
print(public_url)

https://precision-bagged-gem.ngrok-free.dev


In [14]:
%%bash
curl -s https://precision-bagged-gem.ngrok-free.dev/api/chat -d '{
  "model": "qwen2.5:7b",
  "stream": false,
  "messages": [
    { "role": "user", "content": "What is RAG?" }
  ]
}' | jq -r '.message.content'

RAG stands for Retrieval-Augmented Generation. It's a technique used in natural language processing (NLP) and artificial intelligence to enhance the capabilities of language models. Here’s a breakdown of what it entails:

1. **Retrieval**: This part involves fetching relevant information from a database or knowledge base based on the input query or question.

2. **Augmentation**: The retrieved information is then used to augment or enrich the response generated by the language model. This means that the model not only uses its internal knowledge but also incorporates external, contextually relevant data.

3. **Generation**: Finally, a language model generates the output based on both its own knowledge and the additional context provided by the retrieval step.

RAG systems are particularly useful in scenarios where the models need to provide accurate information about specific topics or entities that may not be part of their training data but can be sourced from external databases. This